Парсинг постов по keywords

In [24]:
import csv
import json
import os
import random
import re
import time
from datetime import datetime, timezone
from urllib.parse import quote

import requests
from lxml import html
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# =========================================================
# 1. Константы и настройки
# =========================================================

CSV_FILE = "pikabu_posts.csv"
STATE_FILE = "parsed_state.json"

DATE_FROM = datetime(2020, 1, 1, tzinfo=timezone.utc)

KEYWORDS = [
 "иммунитет", "антитела", "Pfizer",
    "антивакцинация", "непривитые", "анти-ваксеры", "прививка от гриппа"
]

FIELDNAMES = [
    "post_id", "title", "url", "date", "upvotes", "downvotes",
    "comments_count", "reply_posts_count", "text", "keyword"
]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Connection": "keep-alive",
}

# Осторожные паузы:
# между страницами - побольше
# между открытием постов - тоже не слишком часто
SEARCH_PAGE_DELAY_RANGE = (6, 11)
POST_PAGE_DELAY_RANGE = (3, 6)
ERROR_DELAY_RANGE = (20, 40)

# Если видим 403/429 несколько раз подряд, лучше остановиться
MAX_BLOCK_SIGNALS_PER_KEYWORD = 2

In [2]:
# =========================================================
# 2. Вспомогательные функции
# =========================================================

def random_sleep(delay_range: tuple[int, int]) -> None:
    time.sleep(random.uniform(*delay_range))


def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def safe_int(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def ensure_csv_exists(csv_file: str = CSV_FILE) -> None:
    if os.path.exists(csv_file):
        return

    with open(csv_file, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()


def save_post_to_csv(post: dict, csv_file: str = CSV_FILE) -> None:
    with open(csv_file, "a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writerow(post)

In [3]:
# =========================================================
# 3. Работа с состоянием
# =========================================================

def load_state():
    if not os.path.exists(STATE_FILE):
        return set(), {}

    try:
        with open(STATE_FILE, "r", encoding="utf-8") as f:
            content = f.read().strip()
            if not content:
                return set(), {}

            state = json.loads(content)
            parsed_post_ids = set(state.get("parsed_post_ids", []))
            last_page = state.get("last_page", {})
            return parsed_post_ids, last_page

    except json.JSONDecodeError:
        print("⚠️ Файл состояния поврежден или пуст. Начинаем с нуля.")
        return set(), {}


def save_state(parsed_post_ids: set, last_page: dict) -> None:
    state = {
        "parsed_post_ids": list(parsed_post_ids),
        "last_page": last_page
    }
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

In [4]:
# =========================================================
# 4. Session и безопасные запросы
# =========================================================

def build_session() -> requests.Session:
    session = requests.Session()
    session.headers.update(HEADERS)

    retry = Retry(
        total=4,
        backoff_factor=2,
        status_forcelist=[500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )

    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session


def get_with_care(session: requests.Session, url: str, timeout: int = 20):
    """
    Аккуратный GET:
    - не прыгает сразу на следующую страницу при ошибке
    - делает несколько ручных попыток
    - при 403/429 возвращает специальный статус
    """
    max_attempts = 4

    for attempt in range(1, max_attempts + 1):
        try:
            response = session.get(url, timeout=timeout)

            if response.status_code == 200:
                return response, "ok"

            if response.status_code in (403, 429):
                print(f"⚠️ Получен статус {response.status_code} для {url}")
                return response, "blocked"

            print(f"⚠️ Статус {response.status_code} для {url}, попытка {attempt}/{max_attempts}")

        except requests.RequestException as e:
            print(f"⚠️ Ошибка запроса: {e} | попытка {attempt}/{max_attempts}")

        # чем дальше попытка, тем дольше пауза
        sleep_seconds = random.uniform(10, 20) * attempt
        time.sleep(sleep_seconds)

    return None, "failed"


In [5]:
# =========================================================
# 5. Получение текста поста
# =========================================================

def get_post_text(session: requests.Session, url: str) -> str:
    response, status = get_with_care(session, url, timeout=20)

    if status != "ok" or response is None:
        return ""

    try:
        tree = html.fromstring(response.text)
        content_block = tree.xpath('//div[contains(@class, "story__content-inner")]')
        if not content_block:
            return ""

        block = content_block[0]
        texts = block.xpath('.//text()[not(parent::script) and not(parent::style)]')
        text_clean = " ".join(t.strip() for t in texts if t.strip())
        return clean_text(text_clean)

    except Exception as e:
        print(f"⚠️ Ошибка при разборе текста поста {url}: {e}")
        return ""


# =========================================================
# 6. Парсинг карточки поста
# =========================================================

def parse_post_card(post_element, session: requests.Session, keyword: str):
    # POST ID
    post_id_raw = post_element.xpath('./@data-story-id')
    post_id = safe_int(post_id_raw[0], None) if post_id_raw else None
    if not post_id:
        return None

    # DATE
    date_raw = post_element.xpath('.//time/@datetime')
    if not date_raw:
        return None

    try:
        post_date = datetime.fromisoformat(date_raw[0].replace("Z", "+00:00"))
    except ValueError:
        return None

    # Нужны только посты с 2020 года
    if post_date < DATE_FROM:
        return None

    # TITLE + URL
    title_node = post_element.xpath('.//h2[contains(@class, "story__title")]/a')
    if not title_node:
        return None

    title = clean_text(title_node[0].text_content())
    url_rel = title_node[0].get("href")

    if not url_rel:
        return None

    url_full = "https://pikabu.ru" + url_rel if url_rel.startswith("/") else url_rel

    # UPVOTES / DOWNVOTES
    upvotes = 0
    downvotes = 0

    aria = post_element.xpath('.//div[contains(@class,"story__rating-count")]/@aria-label')
    if aria:
        m_plus = re.search(r'(\d+)\s*плю', aria[0], flags=re.IGNORECASE)
        m_minus = re.search(r'(\d+)\s*мин', aria[0], flags=re.IGNORECASE)

        if m_plus:
            upvotes = safe_int(m_plus.group(1), 0)
        if m_minus:
            downvotes = safe_int(m_minus.group(1), 0)
    else:
        pluses = post_element.xpath('.//div[contains(@class,"story__rating-block")]/@data-pluses')
        minuses = post_element.xpath('.//div[contains(@class,"story__rating-block")]/@data-minuses')

        if pluses:
            upvotes = abs(safe_int(pluses[0], 0))
        if minuses:
            downvotes = abs(safe_int(minuses[0], 0))

    # COMMENTS COUNT
    comments_attr = post_element.xpath('./@data-comments')
    comments_count = safe_int(comments_attr[0], 0) if comments_attr else 0

    # REPLY POSTS COUNT
    reply_posts_raw = post_element.xpath('.//a[contains(@class,"story__title-comstories")]/text()')
    reply_posts_count = safe_int(reply_posts_raw[0].strip(), 0) if reply_posts_raw else 0

    # TEXT
    text = get_post_text(session, url_full)
    random_sleep(POST_PAGE_DELAY_RANGE)

    post_data = {
        "post_id": post_id,
        "title": title,
        "url": url_full,
        "date": post_date.isoformat(),
        "upvotes": upvotes,
        "downvotes": downvotes,
        "comments_count": comments_count,
        "reply_posts_count": reply_posts_count,
        "text": text,
        "keyword": keyword,
    }

    return post_data


In [6]:

# =========================================================
# 7. Парсинг одного keyword
# =========================================================

def parse_keyword(session: requests.Session, keyword: str, parsed_post_ids: set, last_page_by_keyword: dict):
    print(f"\n🔍 Ключевое слово: {keyword}")

    page = last_page_by_keyword.get(keyword, 1)
    block_signals = 0

    while True:
        search_url = f"https://pikabu.ru/search?q={quote(keyword)}&st=3&page={page}"
        print(f"📄 Страница {page}: {search_url}")

        response, status = get_with_care(session, search_url, timeout=20)

        if status == "blocked":
            block_signals += 1
            print(f"⚠️ Похоже на ограничение доступа. Сигналов блокировки: {block_signals}")

            if block_signals >= MAX_BLOCK_SIGNALS_PER_KEYWORD:
                print(f"⛔ Останавливаем keyword '{keyword}', чтобы не давить сайт")
                break

            random_sleep(ERROR_DELAY_RANGE)
            continue

        if status == "failed" or response is None:
            print("⚠️ Не удалось загрузить страницу после нескольких попыток.")
            print("Останавливаем текущий keyword, чтобы потом безопасно продолжить.")
            break

        block_signals = 0

        try:
            tree = html.fromstring(response.text)
        except Exception as e:
            print(f"⚠️ Ошибка разбора HTML страницы {page}: {e}")
            break

        posts = tree.xpath('//article[contains(@class,"story") and @data-story-id]')

        if not posts:
            print("✅ Постов на странице не найдено. Завершаем keyword.")
            break

        print(f"Найдено карточек постов: {len(posts)}")

        new_posts_on_page = 0

        for post_element in posts:
            post_data = parse_post_card(post_element, session, keyword)
            if not post_data:
                continue

            post_id = post_data["post_id"]

            if post_id in parsed_post_ids:
                continue

            save_post_to_csv(post_data)
            parsed_post_ids.add(post_id)
            new_posts_on_page += 1

            save_state(parsed_post_ids, last_page_by_keyword)

        print(f"Сохранено новых постов со страницы: {new_posts_on_page}")

        # Сохраняем как есть, по твоему условию
        last_page_by_keyword[keyword] = page
        save_state(parsed_post_ids, last_page_by_keyword)

        page += 1
        random_sleep(SEARCH_PAGE_DELAY_RANGE)

In [25]:
# =========================================================
# 8. Основной запуск
# =========================================================

def main():
    ensure_csv_exists()

    parsed_post_ids, last_page_by_keyword = load_state()
    session = build_session()

    print(f"Уже обработанных post_id: {len(parsed_post_ids)}")

    for keyword in KEYWORDS:
        parse_keyword(session, keyword, parsed_post_ids, last_page_by_keyword)

    print("\n✅ Сбор постов завершен.")


if __name__ == "__main__":
    main()

Уже обработанных post_id: 4817

🔍 Ключевое слово: иммунитет
📄 Страница 1: https://pikabu.ru/search?q=%D0%B8%D0%BC%D0%BC%D1%83%D0%BD%D0%B8%D1%82%D0%B5%D1%82&st=3&page=1
Найдено карточек постов: 10
Сохранено новых постов со страницы: 7
📄 Страница 2: https://pikabu.ru/search?q=%D0%B8%D0%BC%D0%BC%D1%83%D0%BD%D0%B8%D1%82%D0%B5%D1%82&st=3&page=2
Найдено карточек постов: 9
Сохранено новых постов со страницы: 7
📄 Страница 3: https://pikabu.ru/search?q=%D0%B8%D0%BC%D0%BC%D1%83%D0%BD%D0%B8%D1%82%D0%B5%D1%82&st=3&page=3
Найдено карточек постов: 9
Сохранено новых постов со страницы: 8
📄 Страница 4: https://pikabu.ru/search?q=%D0%B8%D0%BC%D0%BC%D1%83%D0%BD%D0%B8%D1%82%D0%B5%D1%82&st=3&page=4
Найдено карточек постов: 9
Сохранено новых постов со страницы: 6
📄 Страница 5: https://pikabu.ru/search?q=%D0%B8%D0%BC%D0%BC%D1%83%D0%BD%D0%B8%D1%82%D0%B5%D1%82&st=3&page=5
Найдено карточек постов: 9
Сохранено новых постов со страницы: 6
📄 Страница 6: https://pikabu.ru/search?q=%D0%B8%D0%BC%D0%BC%D1%83%D0%BD%D0

In [26]:
import pandas as pd

columns = [
    "post_id", "title", "url", "date", "upvotes", "downvotes",
    "comments_count", "reply_posts_count", "text", "keyword"
]

data = pd.read_csv('pikabu_posts.csv', names=columns, header=None)

In [30]:
data.head(2)

,post_id,title,url,date,upvotes,downvotes,comments_count,reply_posts_count,text,keyword
0,13488271,В России выпустили три тестовые серии вакцины ...,https://pikabu.ru/story/v_rossii_vyipustili_tr...,2025-12-11T13:35:02+03:00,2964,106,278,0,🧬 В России создали и выпустили первые тестовые...,вакцина
1,13483464,Каждому по барокамере!,https://pikabu.ru/story/kazhdomu_po_barokamere...,2025-12-10T09:10:00+03:00,832,58,94,0,Из канала Мем в глаз попал Также в МАХ,вакцина


In [27]:
len(data)

5685

In [28]:
len(data['post_id'].unique())

5394

In [29]:
data['post_id'].nunique()

5394